# AuditorAgent Fine-Tuning Pipeline
## QLoRA Training of Phi-3-mini-4k-instruct for Architectural Governance Auditing

**Author:** Damika | **Date:** March 2026
**Task:** Train a specialised AuditorAgent that produces structured JSON audit reports
**Method:** QLoRA (4-bit NormalFloat quantisation) with completion-only loss masking
**Base Model:** Microsoft Phi-3-mini-4k-instruct (3.8B parameters)

---

### Key Design Decisions

| Decision | Choice | Justification |
|----------|--------|---------------|
| Base model | Phi-3-mini-4k-instruct | 3.8B params fits in 16GB VRAM with QLoRA; strong instruction-following; 4K context sufficient for audit payloads |
| Fine-tuning method | QLoRA (LoRA rank=32, all linear layers) | Trains only 0.88% of parameters; 4-bit quantisation reduces memory from 7.6GB to ~2GB |
| Loss masking | DataCollatorForCompletionOnlyLM | Critical: ensures loss is computed ONLY on the assistant output (audit JSON), not on the input payload |
| Data split | Chain-aware 85/15 | Prevents data leakage by keeping all rounds of the same project in the same split |
| Epochs | 3 | Sufficient for convergence on ~800 rows without overfitting; validated by eval loss plateau |
| Learning rate | 2e-4 with cosine decay | Standard for QLoRA; cosine schedule prevents sudden destabilisation of structured output learning |

---
## Section 1 — Environment Setup

In [ ]:
pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install -q "transformers>=4.43.0,<5.0.0" "peft>=0.12.0"
pip install -q "bitsandbytes>=0.43.0" "accelerate>=0.30.0"
pip install -q datasets scikit-learn matplotlib seaborn pandas numpy scipy
pip install -q "trl==0.8.6"
pip install -q tensorboardX tensorboard

In [ ]:
import os, json, time, hashlib, warnings, random, re, copy
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, AutoConfig,
    BitsAndBytesConfig, TrainingArguments, TrainerCallback,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    PeftModel, TaskType,
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import Dataset
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected.")
print(f"Seed: {SEED}")

In [ ]:
# ===============================================================
# CONFIGURATION
# ===============================================================

DATASET_PATH = "auditor_dataset_fixed.jsonl"  # UPDATE to your file
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

OUTPUT_DIR = Path("training_output")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)
FINAL_MODEL_DIR = OUTPUT_DIR / "auditor_agent_model"
FINAL_MODEL_DIR.mkdir(exist_ok=True)
LOGS_DIR = OUTPUT_DIR / "logs"
LOGS_DIR.mkdir(exist_ok=True)

CONFIG = {
    # LoRA
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    # Training
    "num_epochs": 3,
    "per_device_batch_size": 1,
    "gradient_accumulation": 8,
    "learning_rate": 2e-4,
    "lr_scheduler": "cosine",
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 4096,
    # Split
    "val_ratio": 0.15,
}

for k, v in CONFIG.items():
    print(f"  {k}: {v}")

---
## Section 2 — System Prompt & Data Loading

The system prompt is identical to what `main.py` uses at runtime (lines 528-606).
This ensures the fine-tuned model produces output in the exact format the runtime expects.

In [ ]:
# ===============================================================
# AUDITOR SYSTEM PROMPT (matches main.py exactly)
# ===============================================================

AUDITOR_SYSTEM_PROMPT = """You are the strict architecture auditor.

Audit the architecture plan against:
- frozen confirmed requirements
- rich requirement notes
- cumulative issue ledger
- revision memory
- prior audit history

Main goal:
- First, verify whether previously reported issues were actually fixed.
- Second, identify the most important remaining weaknesses.
- Third, explain clearly why the score stayed the same, improved, or dropped.

Rules:
- Use stable issue IDs whenever the same issue still exists.
- Mark each issue status as one of: unresolved, resolved, downgraded, new.
- Re-check prior unresolved issues before creating new ones.
- If an earlier issue was fixed, keep the same issue ID and mark it resolved.
- If an earlier issue still exists, keep the same issue ID and explain what is still missing.
- Only create a new issue ID if the problem is materially different from previous issues.
- Score the plan against an absolute rubric, not against any approval threshold.
- Do not try to make the plan pass or fail a gate.
- Be willing to score below 9 if the plan has real weaknesses.
- If the score drops, explain the exact reason for the drop.
- If the score does not improve, explain what blocked improvement.
- Prefer the most important unresolved issues over minor nitpicks.
- passed is advisory only; the runtime decides approval.

Return JSON only with:
- thinking_summary
- rubric_scores
- summary
- strengths
- concerns
- blocking_issues
- recommendations
- requirement_conflicts
- issue_updates

rubric_scores must include numeric values from 0 to 10 for:
- requirements_alignment
- architecture_quality
- security
- operability
- internal_consistency

Each requirement_conflicts item must include:
- issue_id
- field
- current_value
- proposed_value
- exact_reason
- severity

Each issue_updates item must include:
- id
- title
- severity
- status
- detail

For each issue_updates.detail:
- State whether the issue was fixed, partially fixed, unchanged, or newly introduced.
- Explain exactly what in the plan caused this judgment.
- If the issue affected the score, explain how.
- If the architect improved one part but created another problem, say that clearly.

recommendations should:
- focus on the next highest-impact fixes
- be specific enough for the architect to act on in the next round
- avoid vague advice like "improve architecture quality"

summary should:
- briefly explain overall quality
- say whether the round meaningfully improved over the prior round
- mention the main reason the score changed or stayed flat"""

print(f"System prompt: {len(AUDITOR_SYSTEM_PROMPT)} characters")

In [ ]:
# ===============================================================
# LOAD DATASET
# ===============================================================

def load_jsonl(filepath):
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows

raw_data = load_jsonl(DATASET_PATH)
print(f"Loaded {len(raw_data)} rows from {DATASET_PATH}")

# Quick overview
case_types = Counter(r.get('metadata', {}).get('case_type', '?') for r in raw_data)
pclasses = Counter(r.get('profile', {}).get('projectclass', '?') for r in raw_data)
print(f"Case types: {dict(case_types)}")
print(f"Project classes ({len(pclasses)}): {dict(pclasses)}")

---
## Section 3 — Tokenizer & Data Formatting

**Critical design decision:** We format each training example as a complete text string
using Phi-3's chat template, then use `DataCollatorForCompletionOnlyLM` to mask the loss
so it is computed ONLY on the assistant response.

Without this masking, the model learns to predict the massive input payload (easy, low loss)
instead of learning the audit output format (harder, but what we actually need).

In [ ]:
# ===============================================================
# LOAD TOKENIZER
# ===============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab: {tokenizer.vocab_size}, Pad: {tokenizer.pad_token}, EOS: {tokenizer.eos_token}")

In [ ]:
# ===============================================================
# FORMAT DATASET AS TEXT WITH CHAT TEMPLATE
# ===============================================================

def get_nested(d, *keys, default=None):
    for k in keys:
        if isinstance(d, dict) and k in d:
            return d[k]
    return default

def format_row_as_text(row):
    payload = row.get('input_payload', {})
    target = row.get('target_output', {})

    auditor_input = {
        "round": payload.get('round', 1),
        "frozen_requirement_contract": get_nested(payload,
            'frozenrequirementcontract', 'frozen_requirement_contract', default={}),
        "requirements": payload.get('requirements', {}),
        "accepted_exceptions": get_nested(payload,
            'acceptedexceptions', 'accepted_exceptions', default={}),
        "issue_ledger": get_nested(payload,
            'issueledger', 'issue_ledger', default={}),
        "revision_memory": get_nested(payload,
            'revisionmemory', 'revision_memory', default={}),
        "previous_audits": get_nested(payload,
            'previousaudits', 'previous_audits', default=[]),
        "reasoner_reviews": get_nested(payload,
            'reasonerreviews', 'reasoner_reviews', default={}),
        "specialist_subplans": get_nested(payload,
            'specialistsubplans', 'specialist_subplans', default={}),
        "plan": payload.get('plan', {}),
        "best_audit": get_nested(payload,
            'bestaudit', 'best_audit', default={}),
    }

    messages = [
        {"role": "system", "content": AUDITOR_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(auditor_input, indent=1, ensure_ascii=False)},
        {"role": "assistant", "content": json.dumps(target, indent=1, ensure_ascii=False)},
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


# Format all rows and compute chain fingerprints
formatted_texts = []
chain_fps = []
sample_ids = []
skipped = 0
token_counts = []

for row in raw_data:
    try:
        text = format_row_as_text(row)
        n_tokens = len(tokenizer.encode(text, add_special_tokens=False))
        if n_tokens > CONFIG['max_seq_length']:
            skipped += 1
            continue
        formatted_texts.append(text)
        token_counts.append(n_tokens)
        sample_ids.append(row.get('sample_id', ''))

        contract = get_nested(row.get('input_payload', {}),
                    'frozenrequirementcontract', 'frozen_requirement_contract', default={})
        fp = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:16]
        chain_fps.append(fp)
    except Exception as e:
        skipped += 1

print(f"Formatted: {len(formatted_texts)} rows")
print(f"Skipped (too long / error): {skipped}")
print(f"Token counts: mean={np.mean(token_counts):.0f}, "
      f"median={np.median(token_counts):.0f}, max={max(token_counts)}")

# ---- CRITICAL VERIFICATION ----
print(f"\n{'='*60}")
print("FORMAT VERIFICATION")
print(f"{'='*60}")

sample = formatted_texts[0]
# Phi-3 chat template uses <|assistant|> to mark assistant turn
assistant_marker = "<|assistant|>"
end_marker = "<|end|>"

if assistant_marker in sample:
    pos = sample.index(assistant_marker)
    total_chars = len(sample)
    input_chars = pos
    output_chars = total_chars - pos
    print(f"  Assistant marker found at position {pos}")
    print(f"  Input portion:  {input_chars} chars ({input_chars/total_chars*100:.0f}%)")
    print(f"  Output portion: {output_chars} chars ({output_chars/total_chars*100:.0f}%)")

    # Show the start of assistant content
    assistant_content_start = sample[pos:pos+300]
    print(f"\n  Assistant response starts with:")
    print(f"  {assistant_content_start[:200]}...")

    # Check that the output contains expected keys
    assistant_text = sample[pos:]
    expected_keys = ["thinking_summary", "rubric_scores", "summary",
                     "strengths", "blocking_issues", "issue_updates"]
    found = [k for k in expected_keys if k in assistant_text]
    missing = [k for k in expected_keys if k not in assistant_text]
    print(f"\n  Expected keys found in output: {found}")
    if missing:
        print(f"  WARNING - Missing keys: {missing}")
    else:
        print(f"  All expected keys present in output.")
else:
    print("  CRITICAL ERROR: <|assistant|> marker not found in formatted text!")
    print("  The chat template is not working correctly.")

---
## Section 4 — Completion-Only Loss Masking

**This is the most critical component of the training pipeline.**

`DataCollatorForCompletionOnlyLM` masks the loss on all tokens before the assistant response.
Without this, the model achieves low loss by predicting the repetitive input payload
instead of learning the structured audit output format.

Expected masking ratio: ~80-90% masked (input), ~10-20% unmasked (output).

In [ ]:
# ===============================================================
# COMPLETION-ONLY DATA COLLATOR
# ===============================================================

# Find the response template tokens
response_template = "<|assistant|>"
response_template_ids = tokenizer.encode(
    response_template, add_special_tokens=False
)

print(f"Response template: '{response_template}'")
print(f"Template token IDs: {response_template_ids}")

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

# ---- VERIFY MASKING WORKS ----
print(f"\n{'='*60}")
print("LOSS MASKING VERIFICATION")
print(f"{'='*60}")

# Test on first 5 examples
for i in range(min(5, len(formatted_texts))):
    enc = tokenizer(formatted_texts[i], return_tensors="pt",
                    truncation=True, max_length=CONFIG['max_seq_length'])
    batch = collator([{
        "input_ids": enc["input_ids"][0],
        "attention_mask": enc["attention_mask"][0]
    }])
    labels = batch["labels"][0]
    masked = (labels == -100).sum().item()
    unmasked = (labels != -100).sum().item()
    total = len(labels)
    print(f"  Example {i}: {total} tokens, "
          f"masked={masked} ({masked/total*100:.0f}%), "
          f"unmasked={unmasked} ({unmasked/total*100:.0f}%)")

avg_unmasked_pct = []
for i in range(min(20, len(formatted_texts))):
    enc = tokenizer(formatted_texts[i], return_tensors="pt",
                    truncation=True, max_length=CONFIG['max_seq_length'])
    batch = collator([{
        "input_ids": enc["input_ids"][0],
        "attention_mask": enc["attention_mask"][0]
    }])
    labels = batch["labels"][0]
    total = len(labels)
    unmasked = (labels != -100).sum().item()
    avg_unmasked_pct.append(unmasked / total * 100)

mean_pct = np.mean(avg_unmasked_pct)
print(f"\n  Average unmasked: {mean_pct:.1f}%")
if mean_pct < 5:
    print("  CRITICAL: Less than 5% unmasked - masking may be broken!")
elif mean_pct > 50:
    print("  WARNING: More than 50% unmasked - masking may not be working!")
else:
    print(f"  GOOD: ~{mean_pct:.0f}% of tokens are output (loss computed on these)")

---
## Section 5 — Chain-Aware Train/Validation Split

Rows within the same audit chain share the same project. Splitting at chain level
prevents data leakage between train and validation sets.

In [ ]:
# ===============================================================
# CHAIN-AWARE SPLIT
# ===============================================================

chains = defaultdict(list)
for i, fp in enumerate(chain_fps):
    chains[fp].append(i)

chain_ids = list(chains.keys())
rng = random.Random(SEED)
rng.shuffle(chain_ids)

n_val = max(1, int(len(chain_ids) * CONFIG['val_ratio']))
val_chain_set = set(chain_ids[:n_val])

train_idx, val_idx = [], []
for cid in chain_ids:
    target = val_idx if cid in val_chain_set else train_idx
    target.extend(chains[cid])

train_texts = [formatted_texts[i] for i in train_idx]
val_texts = [formatted_texts[i] for i in val_idx]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

# Verify no leakage
train_chains = set(chain_fps[i] for i in train_idx)
val_chains = set(chain_fps[i] for i in val_idx)
leakage = train_chains & val_chains

print(f"Chains: {len(chains)} total, {len(chains) - n_val} train, {n_val} val")
print(f"Rows: {len(train_texts)} train, {len(val_texts)} val")
print(f"Chain leakage: {'NONE (good)' if not leakage else f'DETECTED: {len(leakage)} chains!'}")

---
## Section 6 — Model Loading with 4-bit Quantisation

NormalFloat4 quantisation reduces the 3.8B parameter model from ~7.6GB to ~2GB,
enabling training on consumer GPUs with 16GB VRAM.

In [ ]:
# ===============================================================
# QUANTIZATION CONFIG
# ===============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ===============================================================
# LOAD MODEL (with rope_scaling fix)
# ===============================================================

config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.rope_scaling = None  # Fix for Phi-3 + latest transformers

print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
)

model.config.use_cache = False
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## Section 7 — LoRA Adapter Configuration

| Parameter | Value | Justification |
|-----------|-------|---------------|
| Rank (r) | 32 | Higher than typical 8-16 because generating structured JSON requires learning specific token patterns for keys, scores, and template formats |
| Alpha | 64 | Standard ratio alpha = 2r for stable gradient scaling |
| Dropout | 0.05 | Light regularisation for small dataset (~800 rows) |
| Target modules | All 7 linear layers | Full expressiveness needed because the task fundamentally changes the output distribution from natural language to structured JSON |

In [ ]:
# ===============================================================
# APPLY LoRA
# ===============================================================

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = trainable / total * 100

print(f"Trainable: {trainable:,} / {total:,} ({pct:.2f}%)")
model.print_trainable_parameters()

---
## Section 8 — Training Configuration & Execution

Training loss will be higher than the previous run (expected ~2-4 vs ~0.13) because
loss is now computed ONLY on the assistant output tokens. This is correct behaviour:
the model is learning the hard part (producing audit JSON) instead of the easy part
(predicting repetitive input payloads).

In [ ]:
# ===============================================================
# TRAINING ARGS
# ===============================================================

total_steps = len(train_dataset) * CONFIG['num_epochs'] // CONFIG['gradient_accumulation']
eval_steps = max(1, int(total_steps * 0.1))
save_steps = eval_steps * 3

print(f"Training plan:")
print(f"  Examples: {len(train_dataset)} train, {len(val_dataset)} val")
print(f"  Epochs: {CONFIG['num_epochs']}, Effective batch: {CONFIG['gradient_accumulation']}")
print(f"  Steps: ~{total_steps}, Eval every {eval_steps}, Save every {save_steps}")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation'],
    learning_rate=CONFIG['learning_rate'],
    lr_scheduler_type=CONFIG['lr_scheduler'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=False,
    bf16=True,
    logging_dir=str(LOGS_DIR),
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="tensorboard",
    seed=SEED,
)
print("Training args ready.")

In [ ]:
# ===============================================================
# METRICS CALLBACK
# ===============================================================

class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.train_losses, self.eval_losses = [], []
        self.learning_rates, self.steps, self.eval_steps_list = [], [], []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                self.train_losses.append(logs['loss'])
                self.steps.append(state.global_step)
            if 'learning_rate' in logs:
                self.learning_rates.append(logs['learning_rate'])
            if 'eval_loss' in logs:
                self.eval_losses.append(logs['eval_loss'])
                self.eval_steps_list.append(state.global_step)

metrics_cb = MetricsCallback()

In [ ]:
# ===============================================================
# TRAIN WITH COMPLETION-ONLY LOSS
# ===============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=collator,
    max_seq_length=CONFIG['max_seq_length'],
    dataset_text_field="text",
    packing=False,
    callbacks=[metrics_cb],
)

print("Starting training with completion-only loss masking...")
print("=" * 60)

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print("=" * 60)
print(f"TRAINING COMPLETE")
print(f"  Duration: {elapsed/60:.1f} minutes")
print(f"  Final loss: {train_result.training_loss:.4f}")
print(f"  Steps: {train_result.global_step}")

trainer.save_metrics("train", train_result.metrics)

In [ ]:
# ===============================================================
# EVALUATE
# ===============================================================

# Fix NotebookProgress callback issue
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if "NotebookProgress" not in type(cb).__name__
]

eval_results = trainer.evaluate()
eval_loss = eval_results.get('eval_loss', 0)
perplexity = np.exp(eval_loss) if eval_loss < 20 else float('inf')

print(f"Eval loss: {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")
trainer.save_metrics("eval", eval_results)

---
## Section 9 — Save Model

**Important:** Save the LoRA adapter BEFORE calling merge_and_unload().

In [ ]:
# ===============================================================
# SAVE LoRA ADAPTER
# ===============================================================

adapter_dir = FINAL_MODEL_DIR / "lora_adapter"
adapter_dir.mkdir(exist_ok=True)

model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

adapter_size = sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6
print(f"Adapter saved: {adapter_dir} ({adapter_size:.1f} MB)")

---
## Section 10 — Quick Generation Test

Verify the model produces the correct audit JSON format before running full evaluation.

In [ ]:
# ===============================================================
# QUICK GENERATION TEST
# ===============================================================

model.eval()

# Use a real example from the dataset
test_row = raw_data[0]
payload = test_row.get('input_payload', {})

test_input = {
    "round": payload.get('round', 1),
    "frozen_requirement_contract": get_nested(payload,
        'frozenrequirementcontract', 'frozen_requirement_contract', default={}),
    "requirements": payload.get('requirements', {}),
    "accepted_exceptions": get_nested(payload,
        'acceptedexceptions', 'accepted_exceptions', default={}),
    "issue_ledger": get_nested(payload,
        'issueledger', 'issue_ledger', default={}),
    "revision_memory": get_nested(payload,
        'revisionmemory', 'revision_memory', default={}),
    "previous_audits": get_nested(payload,
        'previousaudits', 'previous_audits', default=[]),
    "plan": payload.get('plan', {}),
}

messages = [
    {"role": "system", "content": AUDITOR_SYSTEM_PROMPT},
    {"role": "user", "content": json.dumps(test_input, indent=1, ensure_ascii=False)},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=CONFIG['max_seq_length'] - 800)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

print(f"Input tokens: {inputs['input_ids'].shape[1]}")
print("Generating...")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=800,
        temperature=0.1,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

gen_ids = out[0][inputs['input_ids'].shape[1]:]
generated = tokenizer.decode(gen_ids, skip_special_tokens=True)

print(f"\nGenerated ({len(generated)} chars):")
print(generated[:1500])

# Parse check
try:
    start = generated.find('{')
    end = generated.rfind('}')
    if start != -1 and end > start:
        parsed = json.loads(generated[start:end+1])
        print(f"\n{'='*60}")
        print(f"JSON VALID: YES")
        print(f"Keys: {list(parsed.keys())}")
        rubric = parsed.get('rubric_scores', {})
        print(f"Rubric scores: {rubric}")
        issues = parsed.get('issue_updates', [])
        print(f"Issues: {len(issues)}")
        if issues:
            print(f"First issue: {issues[0].get('id','')} - {issues[0].get('title','')}")
    else:
        print(f"\nJSON VALID: NO (no JSON object found)")
except Exception as e:
    print(f"\nJSON VALID: NO ({e})")

---
## Section 11 — Full Structural & Semantic Evaluation

Generate audit outputs for 50 validation examples and measure:
1. **Structural metrics**: JSON validity, schema compliance, rubric ranges, issue template, enum compliance
2. **Semantic metrics**: Rubric score MAE vs reference, issue detection F1, blocking issue agreement

In [ ]:
# ===============================================================
# GENERATE ON VALIDATION SET
# ===============================================================

def generate_audit(model, tokenizer, row, max_new_tokens=800):
    payload = row.get('input_payload', {})
    auditor_input = {
        "round": payload.get('round', 1),
        "frozen_requirement_contract": get_nested(payload,
            'frozenrequirementcontract', 'frozen_requirement_contract', default={}),
        "requirements": payload.get('requirements', {}),
        "accepted_exceptions": get_nested(payload,
            'acceptedexceptions', 'accepted_exceptions', default={}),
        "issue_ledger": get_nested(payload,
            'issueledger', 'issue_ledger', default={}),
        "revision_memory": get_nested(payload,
            'revisionmemory', 'revision_memory', default={}),
        "previous_audits": get_nested(payload,
            'previousaudits', 'previous_audits', default=[]),
        "reasoner_reviews": get_nested(payload,
            'reasonerreviews', 'reasoner_reviews', default={}),
        "specialist_subplans": get_nested(payload,
            'specialistsubplans', 'specialist_subplans', default={}),
        "plan": payload.get('plan', {}),
        "best_audit": get_nested(payload,
            'bestaudit', 'best_audit', default={}),
    }

    messages = [
        {"role": "system", "content": AUDITOR_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(auditor_input, indent=1, ensure_ascii=False)},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                        max_length=CONFIG['max_seq_length'] - max_new_tokens)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )

    gen_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text[3:]
        if text.endswith("```"):
            text = text[:-3]
        text = text.strip()
    try:
        return json.loads(text)
    except:
        pass
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except:
            pass
    return None


# Get validation rows (not formatted texts - we need original rows)
val_sample_ids = set(sample_ids[i] for i in val_idx)
val_rows = [r for r in raw_data if r.get('sample_id', '') in val_sample_ids]

EVAL_SIZE = min(50, len(val_rows))
eval_sample = random.Random(SEED).sample(val_rows, EVAL_SIZE)

print(f"Generating on {EVAL_SIZE} validation examples...")

predictions = []
for i, row in enumerate(eval_sample):
    sid = row.get('sample_id', f'val_{i}')
    start_sample = time.time()

    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{EVAL_SIZE}] {sid}")

    try:
        gen_text = generate_audit(model, tokenizer, row)
        parsed = extract_json(gen_text)
    except Exception as e:
        print(f"  ERROR {sid}: {str(e)[:80]}")
        gen_text = ""
        parsed = None

    predictions.append({
        "sample_id": sid,
        "raw": gen_text[:600],
        "parsed": parsed,
        "reference": row.get('target_output', {}),
        "json_valid": parsed is not None,
    })

    elapsed_sample = time.time() - start_sample
    elapsed_total = time.time() - start_global

    avg_time = elapsed_total / (i + 1)
    remaining = avg_time * (EVAL_SIZE - (i + 1))

    print(f"[{i+1}/{EVAL_SIZE}] {elapsed_sample:.1f}s | ETA: {remaining/60:.1f}m")

    if i < 3:
        print(f"\n  --- {sid} ---")
        print(f"  JSON valid: {parsed is not None}")
        if parsed:
            print(f"  Keys: {list(parsed.keys())[:6]}")
            print(f"  Rubric: {parsed.get('rubric_scores', {})}")
        else:
            print(f"  Raw: {gen_text[:200]}")
        print()

valid = sum(1 for p in predictions if p['json_valid'])
print(f"\nDone. JSON valid: {valid}/{EVAL_SIZE} ({valid/EVAL_SIZE*100:.1f}%)")

In [ ]:
# ===============================================================
# COMPUTE ALL METRICS
# ===============================================================

REQUIRED_KEYS = {"thinking_summary", "rubric_scores", "summary", "strengths",
                 "concerns", "blocking_issues", "recommendations",
                 "requirement_conflicts", "issue_updates"}
RUBRIC_DIMS = ["requirements_alignment", "architecture_quality",
               "security", "operability", "internal_consistency"]
ALLOWED_SEV = {"critical", "high", "medium", "low"}
ALLOWED_STAT = {"unresolved", "resolved", "downgraded", "new"}
DETAIL_RE = re.compile(r"^(Fixed|Partially fixed|Unchanged|Newly introduced)\.?", re.IGNORECASE)

n = len(predictions)
json_valid = sum(1 for p in predictions if p['parsed'] is not None)
schema_ok = 0
rubric_ok = 0
template_ok = 0
template_total = 0
enum_ok = 0
enum_total = 0

rubric_errors = {d: [] for d in RUBRIC_DIMS}
issue_f1s = []
blocking_matches = []

for pred in predictions:
    parsed = pred['parsed']
    ref = pred['reference']
    if not parsed:
        continue

    # Schema
    present_keys = set(parsed.keys())
    if REQUIRED_KEYS.issubset(present_keys):
        schema_ok += 1

    # Rubric range
    rubric = parsed.get('rubric_scores', {})
    if isinstance(rubric, dict):
        all_valid = True
        for d in RUBRIC_DIMS:
            val = rubric.get(d)
            if isinstance(val, (int, float)) and 0 <= val <= 10:
                ref_rubric = ref.get('rubric_scores', ref.get('rubricscores', {}))
                ref_val = ref_rubric.get(d, ref_rubric.get(
                    d.replace('_', ''), None))
                if isinstance(ref_val, (int, float)):
                    rubric_errors[d].append(abs(val - ref_val))
            else:
                all_valid = False
        if all_valid and sum(1 for d in RUBRIC_DIMS if d in rubric) == 5:
            rubric_ok += 1

    # Issue updates
    issues = parsed.get('issue_updates', [])
    if isinstance(issues, list):
        for iu in issues:
            if not isinstance(iu, dict):
                continue
            sev = str(iu.get('severity', '')).lower()
            stat = str(iu.get('status', '')).lower()
            enum_total += 1
            if sev in ALLOWED_SEV and stat in ALLOWED_STAT:
                enum_ok += 1
            detail = str(iu.get('detail', ''))
            template_total += 1
            if DETAIL_RE.match(detail):
                template_ok += 1

    # Issue detection F1
    pred_ids = {str(iu.get('id', '')) for iu in issues if isinstance(iu, dict)}
    ref_issues_list = ref.get('issue_updates', ref.get('issueupdates', []))
    ref_ids = {str(iu.get('id', '')) for iu in (ref_issues_list or []) if isinstance(iu, dict)}
    if ref_ids:
        tp = len(pred_ids & ref_ids)
        prec = tp / max(len(pred_ids), 1)
        rec = tp / max(len(ref_ids), 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-8)
        issue_f1s.append(f1)

    # Blocking agreement
    pb = len(parsed.get('blocking_issues', []) or [])
    rb = len(ref.get('blocking_issues', ref.get('blockingissues', [])) or [])
    blocking_matches.append((pb > 0) == (rb > 0))

# Results
structural = {
    "json_validity_rate": json_valid / n,
    "schema_compliance_rate": schema_ok / n,
    "rubric_range_compliance": rubric_ok / n,
    "issue_template_compliance": template_ok / max(template_total, 1),
    "severity_status_compliance": enum_ok / max(enum_total, 1),
}

print(f"\n{'='*60}")
print(f"STRUCTURAL METRICS (n={n})")
print(f"{'='*60}")
for k, v in structural.items():
    bar = '#' * int(v * 30)
    print(f"  {k:35s}: {v:6.1%}  {bar}")

print(f"\n{'='*60}")
print(f"SEMANTIC METRICS")
print(f"{'='*60}")
overall_mae = []
for d in RUBRIC_DIMS:
    if rubric_errors[d]:
        mae = np.mean(rubric_errors[d])
        overall_mae.extend(rubric_errors[d])
        print(f"  {d:30s} MAE: {mae:.2f}")
    else:
        print(f"  {d:30s} MAE: no data")

if overall_mae:
    print(f"  {'OVERALL':30s} MAE: {np.mean(overall_mae):.2f}")
if issue_f1s:
    print(f"  Issue Detection F1: {np.mean(issue_f1s):.2%}")
if blocking_matches:
    print(f"  Blocking Agreement: {np.mean(blocking_matches):.2%}")

# Save
all_metrics = {
    "training": {
        "train_loss": train_result.training_loss,
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "duration_minutes": elapsed / 60,
        "trainable_params": trainable,
        "trainable_pct": pct,
    },
    "structural": structural,
    "semantic": {
        "rubric_mae": {d: float(np.mean(rubric_errors[d])) if rubric_errors[d] else None for d in RUBRIC_DIMS},
        "rubric_mae_overall": float(np.mean(overall_mae)) if overall_mae else None,
        "issue_f1": float(np.mean(issue_f1s)) if issue_f1s else None,
        "blocking_agreement": float(np.mean(blocking_matches)) if blocking_matches else None,
    }
}
with open(OUTPUT_DIR / 'all_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"\nAll metrics saved to {OUTPUT_DIR / 'all_metrics.json'}")

---
## Section 12 — Results Visualisation

In [ ]:
# ===============================================================
# TRAINING CURVES
# ===============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training loss
if metrics_cb.train_losses:
    axes[0].plot(metrics_cb.steps, metrics_cb.train_losses,
                 alpha=0.4, linewidth=0.8, color='#4C72B0', label='Raw')
    if len(metrics_cb.train_losses) > 10:
        w = min(20, len(metrics_cb.train_losses) // 3)
        smoothed = pd.Series(metrics_cb.train_losses).rolling(w, center=True).mean()
        axes[0].plot(metrics_cb.steps, smoothed, 'r-', linewidth=2, label=f'Smoothed (w={w})')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss (Completion-Only)')
    axes[0].legend()

# Validation loss
if metrics_cb.eval_losses:
    axes[1].plot(metrics_cb.eval_steps_list, metrics_cb.eval_losses,
                 'go-', linewidth=2, markersize=6)
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Validation Loss')

# Learning rate
if metrics_cb.learning_rates:
    lr_steps = metrics_cb.steps[:len(metrics_cb.learning_rates)]
    axes[2].plot(lr_steps, metrics_cb.learning_rates, color='orange', linewidth=2)
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_title('Cosine LR Schedule')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")

In [ ]:
# ===============================================================
# EVALUATION RESULTS
# ===============================================================

fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1. Structural metrics bar chart
ax1 = fig.add_subplot(gs[0, 0])
names = ['JSON\nValid', 'Schema', 'Rubric\nRange', 'Issue\nTemplate', 'Sev/Status\nEnum']
vals = list(structural.values())
colors = ['#55A868' if v > 0.7 else '#DD8452' if v > 0.4 else '#C44E52' for v in vals]
bars = ax1.bar(names, vals, color=colors, edgecolor='black', alpha=0.8)
ax1.set_ylim(0, 1.05)
ax1.set_ylabel('Compliance Rate')
ax1.set_title('Structural Output Quality')
for bar, val in zip(bars, vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.0%}', ha='center', fontsize=9)

# 2. Rubric MAE per dimension
ax2 = fig.add_subplot(gs[0, 1])
dim_short = ['ReqAlign', 'ArchQual', 'Security', 'Operab', 'IntConsist']
dim_maes = [np.mean(rubric_errors[d]) if rubric_errors[d] else 0 for d in RUBRIC_DIMS]
mae_colors = ['#55A868' if m < 1.5 else '#DD8452' if m < 2.5 else '#C44E52' for m in dim_maes]
bars2 = ax2.bar(dim_short, dim_maes, color=mae_colors, edgecolor='black', alpha=0.8)
ax2.set_ylabel('Mean Absolute Error')
ax2.set_title('Rubric Score MAE')
ax2.axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Good (<1.0)')
ax2.axhline(2.0, color='orange', linestyle='--', alpha=0.5, label='Acceptable (<2.0)')
ax2.legend(fontsize=7)
for bar, val in zip(bars2, dim_maes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}', ha='center', fontsize=9)

# 3. Predicted vs Reference rubric scatter
ax3 = fig.add_subplot(gs[0, 2])
pred_scores, ref_scores = [], []
for pred in predictions:
    if pred['parsed'] and pred['reference']:
        p_rub = pred['parsed'].get('rubric_scores', {})
        r_rub = pred['reference'].get('rubric_scores',
                 pred['reference'].get('rubricscores', {}))
        for d in RUBRIC_DIMS:
            pv = p_rub.get(d)
            rv = r_rub.get(d, r_rub.get(d.replace('_', ''), None))
            if isinstance(pv, (int, float)) and isinstance(rv, (int, float)):
                pred_scores.append(pv)
                ref_scores.append(rv)

if pred_scores:
    ax3.scatter(ref_scores, pred_scores, alpha=0.3, s=20, c='#8172B3')
    ax3.plot([0, 10], [0, 10], 'r--', alpha=0.5, label='Perfect')
    ax3.set_xlabel('Reference Score')
    ax3.set_ylabel('Predicted Score')
    ax3.set_title('Predicted vs Reference Rubric')
    ax3.set_xlim(-0.5, 10.5)
    ax3.set_ylim(-0.5, 10.5)
    if len(pred_scores) > 2:
        r_val, p_val = stats.pearsonr(ref_scores, pred_scores)
        ax3.annotate(f'r={r_val:.3f}', xy=(0.05, 0.92), xycoords='axes fraction')
    ax3.legend()

# 4. Issue F1 distribution
ax4 = fig.add_subplot(gs[1, 0])
if issue_f1s:
    ax4.hist(issue_f1s, bins=10, edgecolor='black', alpha=0.7, color='#4C72B0')
    ax4.axvline(np.mean(issue_f1s), color='red', linestyle='--',
                label=f'Mean={np.mean(issue_f1s):.2f}')
    ax4.set_xlabel('F1 Score')
    ax4.set_ylabel('Count')
    ax4.set_title('Issue Detection F1')
    ax4.legend()
else:
    ax4.text(0.5, 0.5, 'No issue data', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Issue Detection F1')

# 5. Summary metrics table
ax5 = fig.add_subplot(gs[1, 1:])
ax5.axis('off')
summary_rows = [
    ['Training Loss', f'{train_result.training_loss:.4f}'],
    ['Eval Loss', f'{eval_loss:.4f}'],
    ['Perplexity', f'{perplexity:.2f}'],
    ['Training Time', f'{elapsed/60:.1f} min'],
    ['Trainable Params', f'{trainable:,} ({pct:.2f}%)'],
    ['JSON Validity', f'{structural["json_validity_rate"]:.1%}'],
    ['Schema Compliance', f'{structural["schema_compliance_rate"]:.1%}'],
    ['Rubric MAE', f'{np.mean(overall_mae):.2f}' if overall_mae else 'N/A'],
    ['Issue Detection F1', f'{np.mean(issue_f1s):.2%}' if issue_f1s else 'N/A'],
    ['Blocking Agreement', f'{np.mean(blocking_matches):.2%}' if blocking_matches else 'N/A'],
]
table = ax5.table(cellText=summary_rows, colLabels=['Metric', 'Value'],
                   loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
ax5.set_title('Comprehensive Results Summary', fontsize=13, fontweight='bold', pad=20)

plt.savefig(OUTPUT_DIR / 'evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: evaluation_results.png")

---
## Section 13 — Save Merged Model & Training Record

In [ ]:
# ===============================================================
# SAVE MERGED MODEL
# ===============================================================

SAVE_MERGED = True

if SAVE_MERGED:
    print("Merging adapter into base model...")
    merged_dir = FINAL_MODEL_DIR / "merged_model"
    merged_dir.mkdir(exist_ok=True)

    merged_model = model.merge_and_unload()

    # Fix tied_weights_keys compatibility
    for name, module in merged_model.named_modules():
        tied = getattr(module, "_tied_weights_keys", None)
        if isinstance(tied, list):
            module._tied_weights_keys = {k: k for k in tied}

    merged_model.save_pretrained(str(merged_dir), safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(str(merged_dir))

    merged_size = sum(f.stat().st_size for f in merged_dir.rglob('*') if f.is_file()) / 1e9
    print(f"Merged model saved: {merged_dir} ({merged_size:.2f} GB)")
else:
    print("Skipped merged model (SAVE_MERGED=False)")

In [ ]:
# ===============================================================
# SAVE TRAINING RECORD
# ===============================================================

record = {
    "model_name": MODEL_NAME,
    "dataset_path": DATASET_PATH,
    "dataset_size": len(raw_data),
    "formatted_size": len(formatted_texts),
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "config": CONFIG,
    "trainable_parameters": trainable,
    "trainable_percentage": pct,
    "training_loss": train_result.training_loss,
    "eval_loss": eval_loss,
    "perplexity": perplexity,
    "training_duration_minutes": elapsed / 60,
    "structural_metrics": structural,
    "semantic_metrics": all_metrics.get("semantic", {}),
    "completion_only_masking": True,
    "seed": SEED,
}

with open(FINAL_MODEL_DIR / 'training_record.json', 'w') as f:
    json.dump(record, f, indent=2, default=str)

print(f"Training record saved to {FINAL_MODEL_DIR / 'training_record.json'}")
print(f"\n{'='*60}")
print("ALL OUTPUTS:")
print(f"  Adapter:      {FINAL_MODEL_DIR / 'lora_adapter'}")
if SAVE_MERGED:
    print(f"  Merged model: {FINAL_MODEL_DIR / 'merged_model'}")
print(f"  Curves:       {OUTPUT_DIR / 'training_curves.png'}")
print(f"  Evaluation:   {OUTPUT_DIR / 'evaluation_results.png'}")
print(f"  Metrics:      {OUTPUT_DIR / 'all_metrics.json'}")
print(f"  Record:       {FINAL_MODEL_DIR / 'training_record.json'}")
print(f"  TensorBoard:  {LOGS_DIR}")
print(f"{'='*60}")

---
## Summary

This notebook fine-tuned Phi-3-mini-4k-instruct using QLoRA with **completion-only loss masking**
to serve as the AuditorAgent in an architectural governance multi-agent system.

### Key Technical Contributions

1. **Completion-only loss masking** via `DataCollatorForCompletionOnlyLM` ensures the model
   learns to produce structured audit JSON rather than predicting input payloads
2. **Chain-aware data splitting** prevents data leakage between audit rounds of the same project
3. **Comprehensive evaluation** combining structural metrics (JSON validity, schema compliance)
   with semantic metrics (rubric score MAE, issue detection F1)

### How to Load the Model for Inference

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Option A: LoRA adapter (small, needs base model)
base = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
model = PeftModel.from_pretrained(base, "training_output/auditor_agent_model/lora_adapter")

# Option B: Merged model (standalone)
model = AutoModelForCausalLM.from_pretrained("training_output/auditor_agent_model/merged_model")
```